<h1 style="background: linear-gradient(to right, #395761, #7799B6);
           color: white;
           padding: 22px;
           border-left: 10px solid #CEEB13;
           text-align: center;
           font-family: 'Roboto Mono', Menlo, 'Courier New', monospace;
           font-weight: 600;">
  What Is a Reasoning Model?
</h1>


A *reasoning model* is a language model that writes out intermediate steps before it commits to an answer. That one sentence is the whole idea. Everything else in this track, from sampling strategies to reinforcement learning, is about getting a model to produce *useful* intermediate steps and about measuring whether they helped.

This notebook is the orientation for *Post-Training with RL: From Scratch to Managed RL*, a track that builds reasoning capability on top of a small open model, step by step, in plain PyTorch, and ends with the same training loop running on a managed service. The companion notebooks are:

- [Text generation from scratch](./02_TextGeneration.ipynb): how tokens go in and come out
- [Evaluating reasoning with a math verifier](./03_MathVerifier.ipynb): how to measure whether any of it works

Each notebook stands on its own: it loads what it needs and explains what it uses.

By the end of this notebook you will:

1. Know what "reasoning" means for a language model, and what it does not mean,
2. See the training pipeline that produces a conventional LLM and where reasoning methods attach to it,
3. Watch a base model and a reasoning model answer the same questions, side by side, on a real GPU,
4. Understand why reasoning costs more compute and when it is worth paying for.

<div align="center"><img src="assets/roadmap.svg" alt="Five stages: pretrained LLM, evaluation, inference-time methods, training methods, and managed RL at scale, with dashed lines looping back to evaluation" width="1000"/></div>

## `Step 1`: Reasoning means generating the steps

Ask a conventional language model a word problem and it produces an answer. Ask a reasoning model the same thing and it first produces a sequence of intermediate steps, then the answer. The steps may be visible, or they may be wrapped in special tags such as `<think> ... </think>` so that a chat interface can hide them. Either way, the mechanism is identical: **the steps are ordinary generated tokens**. The model is not running a calculator or a logic engine. It is predicting the next token, one at a time, and it has been trained or prompted so that the tokens it predicts first are a working-out rather than a conclusion.

This style of output is usually called *chain of thought*. Engineers also say the model "thinks". Neither phrase claims the model reasons the way a person does. They describe a behavior: spend tokens on the path before the destination. That behavior reliably improves accuracy on tasks with several dependent steps, such as arithmetic, logic puzzles, and code.

<div align="center"><img src="assets/direct-vs-reasoning.svg" alt="The same sticker word problem sent to a conventional LLM, which replies 16, and to a reasoning LLM, which writes three numbered steps inside think tags and replies 23" width="900"/></div>

### Seeing it on a real model

We will use two checkpoints of the same 0.6-billion-parameter architecture throughout this track:

| Name in this track | What it is | Trained to |
|---|---|---|
| **base** | [Qwen3-0.6B-Base](https://huggingface.co/Qwen/Qwen3-0.6B-Base), the pretrained model, nothing else | continue text |
| **reasoning** | [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B), the same model after post-training | answer as an assistant, with a `<think>` block first |

Both load through a small helper package, `reasoning_utils`, that lives next to these notebooks. It holds the transformer implementation and a thin tokenizer wrapper. That code is plumbing, not the lesson, so we import it rather than reading it. The text generation notebook opens the box far enough to see how text comes out of it.

In [1]:
import time

import torch

from reasoning_utils import generate_text, get_device, load_qwen3

device = get_device()
base, base_tok = load_qwen3("base", device=device)
reasoner, reasoner_tok = load_qwen3("reasoning", device=device)
print(f"parameters: {sum(p.numel() for p in base.parameters()) / 1e6:.0f}M")

device: cuda (NVIDIA L40S)


parameters: 596M


In [2]:
def ask(model, tokenizer, prompt, max_new_tokens=600):
    """Generate greedily, print as we go, and return (text, number of generated tokens, seconds)."""
    start = time.time()
    text = generate_text(model, tokenizer, prompt, device, max_new_tokens=max_new_tokens, stream=True)
    seconds = time.time() - start
    n_tokens = len(tokenizer.encode(text, chat_template=False))
    print(f"\n\n[{n_tokens} tokens in {seconds:.1f} s]")
    return text, n_tokens, seconds


problem = (
    "Maya has 24 stickers. She gives a third of them to her brother and then buys 7 more. "
    "How many stickers does Maya have now?"
)

A base model has only ever seen documents, so we hand it something that looks like a document with the answer about to start. The reasoning model expects a chat turn, and its tokenizer wraps the prompt in the chat template for us (`reasoner_tok.chat_template` is `True`).

In [3]:
print("=== base model ===")
base_text, base_n, base_s = ask(base, base_tok, problem + "\nAnswer:", max_new_tokens=60)

=== base model ===
 $$
16
$$

Solve the equation. Check your solution. 12 = 3x
Answer: $$
x=4
$$

Solve the equation. Check your solution. 12 = 3x
Answer: $$
x=4
$$

[60 tokens in 1.3 s]


In [4]:
print("=== reasoning model ===")
reasoner_text, reasoner_n, reasoner_s = ask(reasoner, reasoner_tok, problem, max_new_tokens=1000)

=== reasoning model ===
<think>
Okay, let's see. Maya has 24 stickers. She gives a third to her brother, then buys 7 more. I need to figure out how many stickers she has at the end. Hmm, let me break this down step by step.

First, she starts with 24 stickers. When she gives a third to her brother, I need to find out what a third of 24 is. To do that, I can divide 24 by 3. Let me calculate that. 24 divided by 3 is 8. So, she gives 8 stickers to her brother.

Now, after giving the third to her brother, she has 24 minus 8. Let me do that subtraction. 24 minus 8 is 16. So, she has 16 stickers left after giving the third to her brother.

Then, she buys 7 more stickers. So, she adds 7 to her remaining 16. Let me add those together. 16 plus 7 is 23. So, after giving the third and buying the 7, she has 23 stickers.

Wait, let me check again to make sure I didn't make a mistake. Starting with 24. Third of 24 is 8. Subtract 8: 24 - 8 = 16. Then add 7: 16 + 7 = 23. Yeah, that seems right. So, th

Two things to notice. The base model commits to a number in a handful of tokens, and then keeps going, because nothing told it to stop being a document: it starts inventing the next exercise on the worksheet. When we ran this, it also got the answer wrong, because a single jump from question to number skips the "buys 7 more" step. The reasoning model opens a `<think>` block, restates the problem, works through it, double-checks itself, closes the block, and only then writes the answer. Same architecture, same parameter count, different training. The second output is longer by an order of magnitude, and every one of those extra tokens was a forward pass through the network. Hold that thought for Step 5.

## `Step 2`: Where a conventional LLM comes from

Reasoning methods are not a way to train a language model. They are applied to a model that already exists. So it helps to know what that model went through.

<div align="center"><img src="assets/training-pipeline.svg" alt="Random weights flow into pretraining, then post-training with instruction tuning and preference tuning. A bracket below shows three reasoning routes that attach to the result: inference-time scaling, reinforcement learning, distillation" width="900"/></div>

| Stage | Data | Objective | What you get |
|---|---|---|---|
| Pretraining | trillions of tokens of web text, books, code | predict the next token | a **base model**: fluent, knows a lot, continues whatever you give it |
| Instruction tuning | prompt and response pairs written or curated by people | imitate the response | an assistant that answers requests instead of continuing them |
| Preference tuning | pairs of responses with a human (or model) preference | prefer the better one | the same assistant, but polite, concise, and formatted the way people like |

Pretraining is where almost all of the compute goes: thousands of GPUs for months. The two post-training stages are cheap by comparison and are where the model's *behavior* is shaped. Our **reasoning** checkpoint is a base model that went through post-training which, among other things, taught it to open a `<think>` block.

The difference between a base model and an assistant is easy to see. Give both an instruction and watch who follows it. We switch the reasoning model's thinking off for this one so it answers directly, which shows that the `<think>` block is a behavior we can toggle through the prompt format, not a hard-wired part of the network.

In [5]:
instruction = "Explain in two sentences why the sky is blue."

print("=== base model ===")
ask(base, base_tok, instruction, max_new_tokens=90)

print("\n=== reasoning model, thinking switched off ===")
reasoner_tok.thinking = False
ask(reasoner, reasoner_tok, instruction, max_new_tokens=120)
reasoner_tok.thinking = True  # back to the default for the rest of the notebook

=== base model ===
 The sky appears blue because of the way sunlight interacts with molecules in the Earth's atmosphere. When sunlight enters the atmosphere, it is scattered by tiny particles and molecules, such as nitrogen and oxygen. This scattering process, known as Rayleigh scattering, causes shorter wavelengths of light (like blue) to scatter more than longer wavelengths (like red). As a result, the blue light is scattered in all directions, making the sky appear predominantly blue.

[88 tokens in 1.6 s]

=== reasoning model, thinking switched off ===
The sky appears blue because of the way light interacts with the Earth's atmosphere, where shorter wavelengths of light (blue) are scattered more efficiently than longer wavelengths (red and orange). This scattering causes the blue light to be scattered in all directions, making the sky appear blue.

[55 tokens in 1.0 s]


The base model treats the instruction as the opening line of a document. It may produce a reasonable explanation, but it has no notion of "two sentences" as a constraint it owes you, and it will happily run on or wander into related questions. The post-trained model answers the request as a request. That is what instruction tuning buys.

## `Step 3`: Three routes to better reasoning

Once a model exists, there are three families of methods for making it reason better. They differ in one important way: whether the model's weights change.

```mermaid
flowchart LR
    M[Existing LLM<br/>base or assistant] --> A
    M --> B
    M --> C
    subgraph A[1. Inference-time scaling]
        direction TB
        a1[prompt for step-by-step] --> a2[sample several answers] --> a3[vote or self-refine]
    end
    subgraph B[2. Reinforcement learning]
        direction TB
        b1[generate answers] --> b2[score with a verifier] --> b3[update weights toward high scores]
    end
    subgraph C[3. Distillation]
        direction TB
        c1[collect a strong model's traces] --> c2[supervised fine-tune on them]
    end
    A --> R1[better answers,<br/>same weights]
    B --> R2[a trained<br/>reasoning model]
    C --> R2
    style A fill:#C3D6E7,stroke:#000
    style B fill:#CEEB13,stroke:#000
    style C fill:#C3D6E7,stroke:#000
    style M fill:#395761,stroke:#000,color:#fff
    style R1 fill:#fff,stroke:#000
    style R2 fill:#ECAC45,stroke:#000
```

| Route | Changes weights? | Extra cost at inference | Needs | Where in this track |
|---|---|---|---|---|
| **Inference-time scaling** | no | yes, often several times more tokens per question | nothing but compute | the sampling, voting, and self-refinement notebooks |
| **Reinforcement learning** | yes | only what the learned behavior adds | a reward signal; for math and code, a verifier that checks answers | the GRPO notebooks, first on one GPU, then through a managed training API |
| **Distillation** | yes | none beyond the learned behavior | a stronger model to copy from | the distillation notebook |

Reinforcement learning is highlighted in the figure above because it is the route that produced the current generation of open reasoning models, and the one this track spends the most time on. It does not need human-labeled reasoning traces. It needs a way to tell whether a final answer is right, which is exactly what the evaluation notebook builds.

The from-scratch GRPO loop fits on one GPU because the model is small. The same loop at production scale means sharding a large model across many GPUs, keeping sampling and training in sync, and babysitting the cluster, which is a different project from understanding the algorithm. That is why the roadmap ends with stage 5: running the identical loop through a Tinker-compatible API on Crusoe's managed RL service, where you write the sampling and reward code and the service runs the distributed training. By then you will know exactly what the API is doing for you.

## `Step 4`: Pattern matching versus logical reasoning

A language model learns statistical regularities in text. When it completes "The boiling point of water at sea level is" with "100°C", it is recalling a strong association, not deriving anything. That is **pattern matching**, and it is what pretraining optimizes for.

**Logical reasoning** is different: start from premises, apply rules, arrive at a conclusion that follows from them, and notice when something contradicts. A theorem prover does this deterministically. A language model does not have such a mechanism built in. What it can do is *simulate* the pattern of reasoning it has seen in text, and with the right training that simulation gets good enough to be useful.

A good probe is a syllogism whose first premise is false:

```mermaid
flowchart TD
    P1["Premise 1: all prime numbers are odd"] --> I
    P2["Premise 2: 2 is a prime number"] --> I
    I["Inference: 2 is odd"] --> R
    F["Fact: 2 is even"] --> R
    R["Contradiction: which do you trust,<br/>the premise or the fact?"]
    style P1 fill:#C3D6E7,stroke:#000
    style P2 fill:#C3D6E7,stroke:#000
    style I fill:#99AE88,stroke:#000
    style F fill:#ECAC45,stroke:#000
    style R fill:#CEEB13,stroke:#000,stroke-width:3px
```

Taken only on its premises, the correct answer is "yes, 2 is odd". Taken with world knowledge, a careful reasoner should flag that premise 1 is false: 2 is the one even prime. A pure pattern matcher answers with whichever association is stronger in its training data and never mentions the conflict. Let's see what our two models do.

In [6]:
syllogism = "All prime numbers are odd. 2 is a prime number. Is 2 odd? Answer in one or two sentences."

print("=== base model ===")
ask(base, base_tok, syllogism + "\nAnswer:", max_new_tokens=40)

print("\n=== reasoning model ===")
ask(reasoner, reasoner_tok, syllogism, max_new_tokens=700)

=== base model ===
 Yes

Find the value of the expression. $100 \div 10 \div 10$
Answer: $$
1
$$

Solve the equation. Check your solution

[40 tokens in 0.7 s]

=== reasoning model ===
<think>
Okay, let's see. The question is whether 2 is odd, given that all prime numbers are odd and 2 is a prime number. Hmm.

First, I remember that prime numbers are numbers greater than 1 that have no divisors other than 1 and themselves. So, the definition of a prime number is that it's only divisible by 1 and itself. Now, the user is saying that all prime numbers are odd. That seems to be a given fact. So, if 2 is a prime number, then according to the definition, 2 must be odd. 

Wait, but 2 is even. Oh, right! 2 is even, but it's also a prime number. So, even though 2 is not considered odd in the traditional sense, it's still a prime. The key here is that the statement "all prime numbers are odd" is a given, so even though 2 is even, it's still a prime. Therefore, the answer should be that 2 is eve

('<think>\nOkay, let\'s see. The question is whether 2 is odd, given that all prime numbers are odd and 2 is a prime number. Hmm.\n\nFirst, I remember that prime numbers are numbers greater than 1 that have no divisors other than 1 and themselves. So, the definition of a prime number is that it\'s only divisible by 1 and itself. Now, the user is saying that all prime numbers are odd. That seems to be a given fact. So, if 2 is a prime number, then according to the definition, 2 must be odd. \n\nWait, but 2 is even. Oh, right! 2 is even, but it\'s also a prime number. So, even though 2 is not considered odd in the traditional sense, it\'s still a prime. The key here is that the statement "all prime numbers are odd" is a given, so even though 2 is even, it\'s still a prime. Therefore, the answer should be that 2 is even, but since it\'s a prime, it\'s odd. But the question is phrased as "Is 2 odd?" given the premises. So, based on the premises, 2 is odd. \n\nBut wait, maybe I need to chec

When we ran this, the base model said "Yes" and moved on to inventing unrelated exercises. The reasoning model's `<think>` block did something more interesting: it laid out the premises, derived "2 is odd", then interrupted itself with "wait, but 2 is even", and spent the rest of the block trying to reconcile the two. It noticed the contradiction. It then resolved it the wrong way, concluding that 2 is odd because the premises say so, without ever flagging that the premise was false.

That is a very honest picture of where small reasoning models are. The intermediate steps make the conflict visible, which is already a big improvement over a bare "Yes": you can see exactly where the argument went wrong. But the steps are still a learned imitation of how arguments look in text, and when the imitation runs out, the model falls back on the strongest pattern. Two situations where this breaks most often: genuinely novel logical setups the model has not seen variants of, and long chains with many dependent steps. Both are why the rest of this track spends so much effort on *measuring* rather than trusting.

## `Step 5`: Reasoning has a price

Every generated token is one full forward pass through all 28 layers of the network. A reasoning model that writes 400 tokens of working-out before a 10-token answer has done 40 times the compute of a model that writes the answer alone. Many reasoning workflows then multiply that again: sample eight solutions and vote, or call a verifier and retry.

Let's put numbers on the two answers from Step 1.

In [7]:
rows = [("base", base_n, base_s), ("reasoning", reasoner_n, reasoner_s)]
print(f"{'model':<12}{'tokens':>8}{'seconds':>10}{'tokens/s':>10}")
for name, n, s in rows:
    print(f"{name:<12}{n:>8}{s:>10.1f}{n / s:>10.0f}")
print(f"\nthe reasoning answer used {reasoner_n / base_n:.0f}x the tokens and {reasoner_s / base_s:.0f}x the time")

model         tokens   seconds  tokens/s
base              60       1.3        45
reasoning        627      11.0        57

the reasoning answer used 10x the tokens and 8x the time


Tokens per second is about the same for both models, as it must be: same architecture, same hardware, one forward pass per token. The cost difference is entirely in *how many* tokens get generated. This is why reasoning models are the wrong tool for translation, summarization, or simple lookups, and the right tool for problems where the extra steps change the answer. One of the goals of training a reasoning model well is for it to learn when to think at length and when not to. We will meet that tension again when we add a format reward during reinforcement learning.

## `Step 6`: The plan for this track

The notebooks follow the five stages in the roadmap at the top. Each builds on the ideas of the previous one but runs on its own: every notebook loads its own model and data, so you can open any of them directly.

| Stage | Notebook topic | What you build |
|---|---|---|
| orientation | what is a reasoning model (this one) | the vocabulary and the mental model |
| 1. pretrained LLM | text generation from scratch | the tokenizer round trip, one forward pass, the decoding loop, a KV cache, compilation |
| 2. evaluation | a math verifier | answer extraction, normalization, symbolic comparison, and a benchmark harness that reports accuracy |
| 3. inference-time | sampling and voting | temperature and top-p sampling, chain-of-thought prompting, self-consistency |
| 3. inference-time | self-refinement | token-level confidence, best-of-N, iterative feedback |
| 4. training | GRPO from the loss up | reinforcement learning with verifiable rewards on a single GPU |
| 4. training | stabilizing GRPO | clipped ratios, a KL term, format rewards, and tracking what the policy is doing |
| 4. training | distillation | learning from a stronger model's reasoning traces |
| 5. managed RL | the same loop through a Tinker-compatible API | your sampling and reward code, Crusoe's managed service for the distributed training, no cluster to operate |

The verifier is the hinge. It measures everything that comes after it, and during reinforcement learning it becomes the reward.

## Summary

- A reasoning model generates intermediate steps before its answer. The steps are ordinary tokens; there is no separate reasoning engine.
- Conventional LLMs come out of pretraining (next-token prediction at huge scale) followed by instruction and preference tuning. Reasoning methods attach to the result.
- Three routes improve reasoning: inference-time scaling (no weight changes), reinforcement learning with a reward signal, and distillation from a stronger model.
- What models do today is a learned simulation of reasoning built on pattern matching. It works well on familiar structures and degrades on novel or very long chains, which is why evaluation matters so much.
- Reasoning costs tokens, and tokens cost forward passes. Use it where the steps change the answer.

## Further reading

Many sources shaped this track. These are the ones most worth your time after this notebook:

- [Chain-of-Thought Prompting Elicits Reasoning in Large Language Models](https://arxiv.org/abs/2201.11903) (Wei et al., 2022), where the term comes from
- [Learning to Reason with LLMs](https://openai.com/index/learning-to-reason-with-llms/) (OpenAI, 2024), the o1 announcement that put reasoning models on the map
- [DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning](https://arxiv.org/abs/2501.12948) (DeepSeek, 2025), the open training recipe this track follows in spirit
- [DeepSeekMath](https://arxiv.org/abs/2402.03300) (Shao et al., 2024), where GRPO was introduced
- [Qwen3 Technical Report](https://arxiv.org/abs/2505.09388) (Qwen team, 2025), the model family used here
- [Scaling LLM Test-Time Compute Optimally Can Be More Effective Than Scaling Model Parameters](https://arxiv.org/abs/2408.03314) (Snell et al., 2024), on inference-time scaling
- [Let's Verify Step by Step](https://arxiv.org/abs/2305.20050) (Lightman et al., 2023), on grading reasoning and the origin of the MATH-500 split used in the evaluation notebook

Next: [Text generation from scratch](./02_TextGeneration.ipynb).